# Решение 1
- Feature Engineering
- Quantile loss function 
- log target
- optimized catboost params 

In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_absolute_percentage_error
import sys
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.append(os.getcwd())
SEED = 42
np.random.seed(SEED)
train = pd.read_csv('data/train/train.csv')
val = pd.read_csv('data/train/val.csv')
holdout = pd.read_csv('data/input/holdout.csv')

TARGET_COL = ' target_2 '

for df in [train, val, holdout]:
    df['DIST_TO_ADM_CENTER'] = np.log1p(df['DIST_TO_ADM_CENTER'])
    df['TOTAL_RANK_COMPS_CANNIBALS'] = df['TOTAL_RANK_COMPS_CANNIBALS'].fillna(0).astype(np.int64)
    
    df['families_per_competitor'] = df['HuffFamilies'] / (df['TOTAL_RANK_COMPS_CANNIBALS'] + 1)
    df['attraction_per_square'] = df['TRADING_PATCH_SCORE'] / (df['TRADE_SQUARE'] + 1e-6)
    df['families_x_24h'] = df['HuffFamilies'] * df['ENTIRE_DAY']
    df['traffic_x_24h'] = df['ROUTES_CNT'] * df['ENTIRE_DAY']
    df['families_x_attraction'] = df['HuffFamilies'] * df['TRADING_PATCH_SCORE']

categorical = [
    'REGION', 'BRANCH', 'CITY', 'subject', 'SUBFRMT', 'HOSPITAL',
    'KINDERGARTENS', 'BANKS', 'MK_ON_MM', 'CROSSWALK',
    'CROSSROAD', 'DENSITY_FAMILY_TYPE', 'COLLEGES',
    'IN_COAL_CITY', 'IN_OIL_CITY', 'CITY_TYPE', 'ON_DUPLICATE_ROAD',
    'INSIDE_YARD', 'ON_INSIDE_DISTRICT_ROAD', 'ON_MAIN_CITY_ROAD',
    'ON_INTERCITY_HIGHWAY', 'ENTRANCE_TO_DISTRICT', 'ON_DISTRICT_BOTTOM',
    'HUB', 'ALCOHOL', 'TOBACCO', 'LOCATION_MARKET', 'STATIONS', 'TRC',
    'METRO', 'LOCATION_PARK', 'MINI_TRC', 'TRADING_PATCH', 'ENTIRE_DAY',
    'MORNING_ROADSIDE', 'SEA', 'FLOORS_TZ', 'SNT', 'LOCATION_TYPE',
    'new_buildings', 'change_CA'
]

base_numeric = [
    'TRADE_SQUARE', 'HuffFamilies', 'HuffRelativeFamilies', 'ROUTES_CNT', 
    'huff_hotel', 'TRADING_PATCH_SCORE', 'RENT_HEX', 'HUFF_RANK_COMPS_CANNIBALS', 
    'TOTAL_RANK_COMPS_CANNIBALS', 'DIST_TO_ADM_CENTER', 'PARKING', 'month_count'
]

best_engineered = [
    'families_per_competitor', 'attraction_per_square', 'families_x_24h', 
    'traffic_x_24h', 'families_x_attraction'
]

drop_cols = ['ID', 'target_1', ' target_2 ']
features = [c for c in base_numeric + categorical + best_engineered if c in train.columns and c not in drop_cols]
features = list(dict.fromkeys(features))
cat_features = [c for c in categorical if c in features]

best_params = {
    'loss_function': 'Quantile:alpha=0.25035473489423127',
    'depth': 5,
    'learning_rate': 0.042349436679478714,
    'iterations': 3344,
    'l2_leaf_reg': 0.48192445522872746,
    'random_strength': 4.432552658816445,
    'rsm': 0.879540400085388,
    'bootstrap_type': 'Bernoulli',
    'subsample': 0.6966062704693177,
    'eval_metric': 'MAPE',
    'random_seed': 42,
    'verbose': 0
}

train_pool = Pool(train[features], np.log1p(train[TARGET_COL]), cat_features=cat_features)
val_pool = Pool(val[features], np.log1p(val[TARGET_COL]), cat_features=cat_features)

model_eval = CatBoostRegressor(**best_params)
model_eval.fit(train_pool, eval_set=val_pool, early_stopping_rounds=150)

val_preds = np.expm1(model_eval.predict(val[features]))
val_preds = np.maximum(val_preds, 0)
print(f"Validation MAPE: {mean_absolute_percentage_error(val[TARGET_COL], val_preds)}")

train_full = pd.concat([train, val], axis=0).reset_index(drop=True)
y_full_log = np.log1p(train_full[TARGET_COL])
final_pool = Pool(train_full[features], y_full_log, cat_features=cat_features)

final_model = CatBoostRegressor(**best_params)
final_model.fit(final_pool)

holdout_preds = np.expm1(final_model.predict(holdout[features]))
holdout_preds = np.maximum(holdout_preds, 0)

submission = pd.DataFrame({
    'id': holdout['ID'].values,
    'PREDICT': holdout_preds
})
submission

Validation MAPE: 0.15058042628545862


,id,PREDICT
0,17178,174752.368058
1,17179,227343.010442
2,17180,188180.559380
3,17181,296808.108463
4,17182,191818.832939
...,...,...
1317,18495,212424.384048
1318,18496,296153.395140
1319,18497,195856.755220
1320,18498,160286.525906


# Оптимизация весов признаков

Диапазоны подбора весов и гиперпараметров catboost намеренно узкие, они были установленны исходя из экспериментов с оптимизацией

In [3]:
import optuna
train = pd.read_csv('data/train/train.csv')
val = pd.read_csv('data/train/val.csv')
holdout = pd.read_csv('data/input/holdout.csv')
TARGET_COL = ' target_2 '

for df in [train, val, holdout]:
    df['DIST_TO_ADM_CENTER'] = np.log1p(df['DIST_TO_ADM_CENTER'])
    df['TOTAL_RANK_COMPS_CANNIBALS'] = df['TOTAL_RANK_COMPS_CANNIBALS'].fillna(0).astype(np.int64)
    df['families_per_competitor'] = df['HuffFamilies'] / (df['TOTAL_RANK_COMPS_CANNIBALS'] + 1)
    df['attraction_per_square'] = df['TRADING_PATCH_SCORE'] / (df['TRADE_SQUARE'] + 1e-6)
    df['families_x_24h'] = df['HuffFamilies'] * df['ENTIRE_DAY']
    df['traffic_x_24h'] = df['ROUTES_CNT'] * df['ENTIRE_DAY']
    df['families_x_attraction'] = df['HuffFamilies'] * df['TRADING_PATCH_SCORE']

categorical = [
    'REGION', 'BRANCH', 'CITY', 'subject', 'SUBFRMT', 'HOSPITAL', 'KINDERGARTENS', 'BANKS', 
    'MK_ON_MM', 'CROSSWALK', 'CROSSROAD', 'DENSITY_FAMILY_TYPE', 'COLLEGES', 'IN_COAL_CITY', 
    'IN_OIL_CITY', 'CITY_TYPE', 'ON_DUPLICATE_ROAD', 'INSIDE_YARD', 'ON_INSIDE_DISTRICT_ROAD', 
    'ON_MAIN_CITY_ROAD', 'ON_INTERCITY_HIGHWAY', 'ENTRANCE_TO_DISTRICT', 'ON_DISTRICT_BOTTOM', 
    'HUB', 'ALCOHOL', 'TOBACCO', 'LOCATION_MARKET', 'STATIONS', 'TRC', 'METRO', 'LOCATION_PARK', 
    'MINI_TRC', 'TRADING_PATCH', 'ENTIRE_DAY', 'MORNING_ROADSIDE', 'SEA', 'FLOORS_TZ', 'SNT', 
    'LOCATION_TYPE', 'new_buildings', 'change_CA'
]
base_numeric = ['TRADE_SQUARE', 'HuffFamilies', 'HuffRelativeFamilies', 'ROUTES_CNT', 'huff_hotel', 'TRADING_PATCH_SCORE', 'RENT_HEX', 'HUFF_RANK_COMPS_CANNIBALS', 'TOTAL_RANK_COMPS_CANNIBALS', 'DIST_TO_ADM_CENTER', 'PARKING', 'month_count']
best_engineered = ['families_per_competitor', 'attraction_per_square', 'families_x_24h', 'traffic_x_24h', 'families_x_attraction']

drop_cols = ['ID', 'target_1', ' target_2 ']
features = list(dict.fromkeys([c for c in base_numeric + categorical + best_engineered if c in train.columns and c not in drop_cols]))
cat_features = [c for c in categorical if c in features]

y_train_log = np.log1p(train[TARGET_COL])
y_val_log = np.log1p(val[TARGET_COL])

def objective(trial):
    f_weights = {
        "TOTAL_RANK_COMPS_CANNIBALS": trial.suggest_float("w_trcc", 1.40, 1.50),
        "SUBFRMT": trial.suggest_float("w_sub", 0.82, 0.92),
        "month_count": trial.suggest_float("w_mc", 0.10, 0.20),
        "HuffFamilies": trial.suggest_float("w_hf", 1.65, 1.80),
        "TRADE_SQUARE": trial.suggest_float("w_ts", 0.45, 0.55),
        "subject": trial.suggest_float("w_subj", 1.45, 1.55),
        "HuffRelativeFamilies": trial.suggest_float("w_hrf", 1.18, 1.28)
    }
    
    cb_params = {
        'loss_function': f'Quantile:alpha={trial.suggest_float("alpha", 0.21, 0.24)}',
        'depth': trial.suggest_int("depth", 4, 7),
        'learning_rate': trial.suggest_float("learning_rate", 0.01, 0.05, log=True),
        'l2_leaf_reg': trial.suggest_float("l2_leaf_reg", 0.5, 15.0, log=True),
        'random_strength': trial.suggest_float("random_strength", 1.0, 10.0),
        'rsm': trial.suggest_float("rsm", 0.7, 1.0), 
        'subsample': trial.suggest_float("subsample", 0.6, 0.9), 
        'bootstrap_type': 'Bernoulli',
        'iterations': 1500, 
        'feature_weights': f_weights,
        'eval_metric': 'MAPE',
        'random_seed': 42,
        'verbose': 0
    }
    
    model = CatBoostRegressor(**cb_params)
    model.fit(
        Pool(train[features], y_train_log, cat_features=cat_features), 
        eval_set=Pool(val[features], y_val_log, cat_features=cat_features), 
        early_stopping_rounds=100
    )
    
    preds = np.expm1(model.predict(val[features]))
    return mean_absolute_percentage_error(val[TARGET_COL], np.maximum(preds, 0))

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=120)

best = study.best_params
final_f_weights = {
    "TOTAL_RANK_COMPS_CANNIBALS": best['w_trcc'],
    "SUBFRMT": best['w_sub'],
    "month_count": best['w_mc'],
    "HuffFamilies": best['w_hf'],
    "TRADE_SQUARE": best['w_ts'],
    "subject": best['w_subj'],
    "HuffRelativeFamilies": best['w_hrf']
}

final_cb_params = {
    'loss_function': f"Quantile:alpha={best['alpha']}",
    'depth': best['depth'],
    'learning_rate': best['learning_rate'],
    'l2_leaf_reg': best['l2_leaf_reg'],
    'random_strength': best['random_strength'],
    'rsm': best['rsm'],
    'subsample': best['subsample'],
    'bootstrap_type': 'Bernoulli',
    'iterations': 4000, 
    'feature_weights': final_f_weights,
    'eval_metric': 'MAPE',
    'random_seed': 42,
    'verbose': 100
}

val_model = CatBoostRegressor(**final_cb_params)
val_model.fit(Pool(train[features], y_train_log, cat_features=cat_features), 
              eval_set=Pool(val[features], y_val_log, cat_features=cat_features), 
              early_stopping_rounds=200)

clean_val_preds = np.maximum(np.expm1(val_model.predict(val[features])), 0)
print(f"\nFINAL HONEST VALIDATION MAPE: {mean_absolute_percentage_error(val[TARGET_COL], clean_val_preds)}")

train_full = pd.concat([train, val], axis=0).reset_index(drop=True)
y_full_log = np.log1p(train_full[TARGET_COL])

final_model = CatBoostRegressor(**final_cb_params)
final_model.fit(Pool(train_full[features], y_full_log, cat_features=cat_features))

holdout_preds = np.maximum(np.expm1(final_model.predict(holdout[features])), 0)
pd.DataFrame({'id': holdout['ID'].values, 'PREDICT': holdout_preds}).to_csv('predictions.csv', index=False)

print("Best Parameters Found:", best)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-05-23 21:50:11,753] A new study created in memory with name: no-name-6eb2f3d2-b4f3-4bdf-84d5-ca32427086c5
[I 2026-05-23 21:50:21,225] Trial 0 finished with value: 0.15064148897936627 and parameters: {'w_trcc': 1.4598074667032117, 'w_sub': 0.8380155923450853, 'w_mc': 0.1371201858590528, 'w_hf': 1.751119444350329, 'w_ts': 0.4707479277699373, 'w_subj': 1.5068720826218767, 'w_hrf': 1.2281630861547705, 'alpha': 0.22566505515097957, 'depth': 6, 'learning_rate': 0.03815343632959353, 'l2_leaf_reg': 0.5020787869890104, 'random_strength': 5.959895418904659, 'rsm': 0.9387278591967309, 'subsample': 0.8263539846071165}. Best is trial 0 with value: 0.15064148897936627.
[I 2026-05-23 21:50:31,350] Trial 1 finishe

0:	learn: 0.0217212	test: 0.0178407	best: 0.0178407 (0)	total: 6.79ms	remaining: 27.1s
100:	learn: 0.0163081	test: 0.0140643	best: 0.0140643 (100)	total: 582ms	remaining: 22.5s
200:	learn: 0.0150262	test: 0.0133642	best: 0.0133642 (200)	total: 1.15s	remaining: 21.7s
300:	learn: 0.0144712	test: 0.0130793	best: 0.0130793 (300)	total: 1.77s	remaining: 21.8s
400:	learn: 0.0140269	test: 0.0127730	best: 0.0127730 (400)	total: 2.35s	remaining: 21.1s
500:	learn: 0.0137877	test: 0.0126591	best: 0.0126591 (500)	total: 2.91s	remaining: 20.3s
600:	learn: 0.0136058	test: 0.0125901	best: 0.0125894 (598)	total: 3.49s	remaining: 19.7s
700:	learn: 0.0134801	test: 0.0125515	best: 0.0125513 (699)	total: 4.15s	remaining: 19.5s
800:	learn: 0.0133602	test: 0.0125213	best: 0.0125176 (790)	total: 5.13s	remaining: 20.5s
900:	learn: 0.0132467	test: 0.0124919	best: 0.0124911 (884)	total: 5.83s	remaining: 20s
1000:	learn: 0.0131616	test: 0.0124764	best: 0.0124764 (1000)	total: 6.53s	remaining: 19.6s
1100:	learn: 

Возможно в прошлый раз он нашел получше, мб там случайность, но концептуально все так работало